In [72]:
import xml.etree.ElementTree as ET
import tkinter as tk
from tkinter import ttk
from tkinter import filedialog
from tkinter import messagebox
import re
# Каждая ячейка теперь — это отдельный пустой список []
data = []
item = {}
m = 0
count = 0
print(data)
# Теперь этот код сработает идеально:
#data[0][1].extend([1, 2, 3, 4, 5, 6, 7])
# В этой ячейке вместо [] станет [1, 2, 3, 4, 5, 6, 7]

def recursive_parse(element, level=0):
    global data, m, count
    print(f"0_{element.tag}")
    print(f"1_{element.attrib}")
    print(f"2_{element.get('name')}")
    print(f"3_{element.get('caption')}")
    print(f"4_{element[1][0].tag}")
    print(f"44_{element[1][0].get('name')}")
    print(f"5_{len(element)}")
    print(f"6_{element.tag}")
    #for i in range(len(element)):
        #print(f"--{element[i].tag}")
        #if 'address' in element[i].attrib : 
            #print("8")
            #print(element[i].get('address'))   

    for j in range(len(element)):
        for k in range(len(element[j])):            
            
            at = element[j][k].get('name')         #Аттрибут для определения массива
            tp = element[j][k].get('type')         #Аттрибут для определения типа
            match = re.search(r'^(.+)\[(\d+)\]$', at)   #Поиск символов [] для построения массива

            if  match:                                                   #Если есть символ массива, то через цикл записывает 
                array_name = match.group(1)
                index = int(match.group(2))
                for i in range(index):
                    item = {
                    "count": count,
                    "class": element[j].tag,
                    "name": (f"{array_name}[{i}]"),#"name": element[j][k].get('name'),
                    "type": element[j][k].get('type'),
                    "caption": element[j][k].get('caption'),
                    "value": 0
                    }
                    data.insert(m, item)
                    m += 1      
                    if tp == 'unsigned char' or tp =='signed char' : count += 1
                    elif tp == 'unsigned long' or tp == 'float': count += 4
                    elif tp == 'unsigned int' : count += 2
                    #print(f"{array_name}[{i}]")
                #print(at)
            else:                                                         #Если нет символа массива
                item = {
                    "count": count,
                    "class": element[j].tag,
                    "name": at,#"name": element[j][k].get('name'),
                    "type": element[j][k].get('type'),
                    "caption": element[j][k].get('caption'),
                    "value": 0
                }
                data.insert(m, item)
                m += 1      
                if tp == 'unsigned char' or tp =='signed char' : count += 1
                elif tp == 'unsigned long' or tp == 'float': count += 4
                elif tp == 'unsigned int' : count += 2
    create_table()
def create_table():
    for row in data:
        values = [
            row.get('count', ''),
            row.get('class', ''),
            row.get('name', ''),
            row.get('type', ''),
            row.get('caption', ''),
            row.get('value', '')
        ]
        tree.insert('', tk.END, values=values)
def load_xml_file():
    """Загрузка XML файла через диалоговое окно"""
    filename = filedialog.askopenfilename(
        title="Выберите XML файл",
        filetypes=[
            ("XML files", "*.xml"),
            ("All files", "*.*")
        ],
        initialdir="./"  # Начинаем с текущей директории
    )
        
    if filename:
        current_file = filename
        status_label.config(text=f"Загрузка: {filename}", fg="orange")
        info_label.config(text=f"Файл: {filename}")
        root.update()
            
            # Парсим и заполняем таблицу
        parse_xml_recursive(current_file)
            
        
def parse_xml_recursive(filename):
    """Парсинг XML с рекурсивным обходом"""
    tree = ET.parse(filename)
    root = tree.getroot()
    recursive_parse(root)

parse_xml_recursive('devices/6.15.xml')

# Заголовки колонок                                                          Интерфейс №1

# Создаем окно
root = tk.Tk()
root.title("Таблица из данных")
root.geometry("800x800")

# Создаем таблицу
columns = ['count', 'class', 'name', 'type', 'caption', 'value']
column_labels = {
    'count': '№',
    'class': 'Класс',
    'name': 'Имя',
    'type': 'Тип',
    'caption': 'Описание',
    'value': 'Значение'
}
# Создаем Treeview (таблицу)
tree = ttk.Treeview(root, columns=columns, show='headings')

toolbar = tk.Frame(root, bd=1, relief=tk.RAISED)
toolbar.pack(side=tk.TOP, fill=tk.X, padx=10, pady=5)
load_btn = ttk.Button(toolbar, text="📁 Загрузить XML...", command=parse_xml_recursive('devices/test.xml'))#load_xml_file)
load_btn.pack(side=tk.LEFT, padx=5, pady=5)
# Настраиваем колонки
for col in columns:
    tree.heading(col, text=column_labels[col])
    tree.column(col, width=100, anchor='center')
tree.column('count', width=50)
tree.column('caption', width=200)

# Добавляем скролл
scrollbar = ttk.Scrollbar(root, orient=tk.VERTICAL, command=tree.yview)
tree.configure(yscrollcommand=scrollbar.set)

# Размещаем
tree.pack(side=tk.LEFT, fill=tk.BOTH, expand=True, padx=10,pady=10)
scrollbar.pack(side=tk.RIGHT, fill=tk.Y, pady=10)

# ============================================
# ЗАПОЛНЯЕМ ТАБЛИЦУ
# ============================================
for row in data:
    values = [
        row.get('count', ''),
        row.get('class', ''),
        row.get('name', ''),
        row.get('type', ''),
        row.get('caption', ''),
        row.get('value', '')
    ]
    tree.insert('', tk.END, values=values)

# Информация
label = tk.Label(root, text=f"Всего записей: {len(data)}", font=('Arial', 10))
label.pack(pady=5)

root.mainloop()

[]
0_device
1_{'name': 'IKTS11', 'caption': 'ИКТС-11', 'id': '6', 'build': '15'}
2_IKTS11
3_ИКТС-11
4_item
44_AnySensCalibrFault
5_8
6_device


TclError: invalid command name ".!treeview"

In [59]:
import tkinter as tk
from tkinter import ttk, filedialog
import xml.etree.ElementTree as ET

class SimpleXMLViewer:
    """Простой просмотрщик XML с выбором файла"""
    
    def __init__(self, root):
        self.root = root
        self.root.title("Просмотр XML")
        self.root.geometry("800x500")
        
        # Кнопка
        tk.Button(root, text="Выбрать XML файл", 
                 command=self.load_xml).pack(pady=10)
        
        # Таблица
        self.tree = ttk.Treeview(root, columns=('name', 'type'), show='headings')
        self.tree.heading('name', text='Имя')
        self.tree.heading('type', text='Тип')
        self.tree.pack(fill=tk.BOTH, expand=True, padx=10, pady=10)
        
        # Статус
        self.status = tk.Label(root, text="Выберите XML файл", fg="blue")
        self.status.pack(pady=5)
    
    def load_xml(self):
        filename = filedialog.askopenfilename(filetypes=[("XML", "*.xml")])
        if filename:
            try:
                tree = ET.parse(filename)
                root = tree.getroot()
                
                # Очищаем таблицу
                for item in self.tree.get_children():
                    self.tree.delete(item)
                
                # Заполняем
                for device in root.findall('device'):
                    for param in device:
                        name = param.get('name', '')
                        tp = param.get('type', '')
                        self.tree.insert('', tk.END, values=(name, tp))
                
                self.status.config(text=f"Загружен: {filename}", fg="green")
                
            except Exception as e:
                self.status.config(text=f"Ошибка: {e}", fg="red")

root = tk.Tk()
app = SimpleXMLViewer(root)
root.mainloop()

In [60]:
import tkinter as tk
from tkinter import ttk, filedialog, messagebox
import xml.etree.ElementTree as ET
import re

class XMLTableApp:
    """Приложение для загрузки XML и отображения в таблице"""
    
    def __init__(self, root):
        self.root = root
        self.root.title("XML Viewer с выбором файла")
        self.root.geometry("1000x700")
        
        # Переменные
        self.current_file = None
        self.data = []
        self.tree = None
        self.array_data = {}  # Для хранения массивов
        
        # Создаем интерфейс
        self.create_widgets()
        
        # Статус
        self.status_label.config(text="Загрузите XML файл", fg="blue")
    
    def create_widgets(self):
        """Создание всех виджетов"""
        # ============================================
        # Верхняя панель с кнопками
        # ============================================
        toolbar = tk.Frame(self.root)
        toolbar.pack(fill=tk.X, padx=5, pady=5)
        
        # Кнопка выбора XML файла
        self.load_btn = tk.Button(
            toolbar, 
            text="📁 Выбрать XML файл", 
            command=self.load_xml_file,
            bg="#4CAF50", 
            fg="white",
            font=('Arial', 10, 'bold'),
            padx=10,
            pady=5
        )
        self.load_btn.pack(side=tk.LEFT, padx=5)
        
        # Кнопка обновления
        self.refresh_btn = tk.Button(
            toolbar, 
            text="🔄 Обновить", 
            command=self.refresh_table,
            bg="#2196F3", 
            fg="white",
            padx=10,
            pady=5
        )
        self.refresh_btn.pack(side=tk.LEFT, padx=5)
        
        # Кнопка экспорта
        self.export_btn = tk.Button(
            toolbar, 
            text="📊 Экспорт CSV", 
            command=self.export_csv,
            bg="#FF9800", 
            fg="white",
            padx=10,
            pady=5
        )
        self.export_btn.pack(side=tk.LEFT, padx=5)
        
        # Кнопка очистки
        self.clear_btn = tk.Button(
            toolbar, 
            text="🗑️ Очистить", 
            command=self.clear_all,
            bg="#f44336", 
            fg="white",
            padx=10,
            pady=5
        )
        self.clear_btn.pack(side=tk.LEFT, padx=5)
        
        # Статус
        self.status_label = tk.Label(
            toolbar, 
            text="Готов", 
            font=('Arial', 10),
            fg="green"
        )
        self.status_label.pack(side=tk.RIGHT, padx=10)
        
        # ============================================
        # Информационная панель
        # ============================================
        info_frame = tk.Frame(self.root)
        info_frame.pack(fill=tk.X, padx=10, pady=5)
        
        self.info_label = tk.Label(
            info_frame, 
            text="Файл не загружен", 
            font=('Arial', 9),
            fg="gray"
        )
        self.info_label.pack(side=tk.LEFT)
        
        self.count_label = tk.Label(
            info_frame, 
            text="Записей: 0", 
            font=('Arial', 9),
            fg="gray"
        )
        self.count_label.pack(side=tk.RIGHT)
        
        # ============================================
        # Таблица (Treeview)
        # ============================================
        table_frame = tk.Frame(self.root)
        table_frame.pack(fill=tk.BOTH, expand=True, padx=10, pady=5)
        
        self.create_table(table_frame)
        
        # ============================================
        # Нижняя панель с дополнительной информацией
        # ============================================
        bottom_frame = tk.Frame(self.root)
        bottom_frame.pack(fill=tk.X, padx=10, pady=5)
        
        self.detail_label = tk.Label(
            bottom_frame, 
            text="", 
            font=('Arial', 9),
            fg="gray"
        )
        self.detail_label.pack(side=tk.LEFT)
    
    def create_table(self, parent):
        """Создание таблицы Treeview"""
        # Определяем колонки
        self.columns = ('count', 'class', 'name', 'type', 'caption', 'value', 'is_array')
        column_labels = {
            'count': '№',
            'class': 'Класс',
            'name': 'Имя',
            'type': 'Тип',
            'caption': 'Описание',
            'value': 'Значение',
            'is_array': 'Массив'
        }
        
        # Создаем Treeview
        self.tree = ttk.Treeview(parent, columns=self.columns, show='headings')
        
        # Настраиваем колонки
        for col in self.columns:
            self.tree.heading(col, text=column_labels.get(col, col))
            width = 50 if col == 'count' else 80 if col == 'is_array' else 150
            self.tree.column(col, width=width, anchor='center')
        
        # Увеличиваем ширину для некоторых колонок
        self.tree.column('caption', width=200)
        self.tree.column('name', width=150)
        
        # Добавляем скролл
        scrollbar = ttk.Scrollbar(parent, orient=tk.VERTICAL, command=self.tree.yview)
        self.tree.configure(yscrollcommand=scrollbar.set)
        
        # Размещаем
        self.tree.pack(side=tk.LEFT, fill=tk.BOTH, expand=True)
        scrollbar.pack(side=tk.RIGHT, fill=tk.Y)
        
        # Привязываем событие выбора строки
        self.tree.bind('<<TreeviewSelect>>', self.on_item_select)
    
    def load_xml_file(self):
        """Загрузка XML файла через диалоговое окно"""
        filename = filedialog.askopenfilename(
            title="Выберите XML файл",
            filetypes=[
                ("XML files", "*.xml"),
                ("All files", "*.*")
            ],
            initialdir="./"  # Начинаем с текущей директории
        )
        
        if filename:
            self.current_file = filename
            self.status_label.config(text=f"Загрузка: {filename}", fg="orange")
            self.info_label.config(text=f"Файл: {filename}")
            self.root.update()
            
            # Парсим и заполняем таблицу
            success = self.parse_xml(filename)
            
            if success:
                self.status_label.config(text=f"✅ Загружен: {filename}", fg="green")
            else:
                self.status_label.config(text="❌ Ошибка загрузки", fg="red")
    
    def parse_xml(self, filename):
        """
        Парсинг XML файла и заполнение данных
        
        Returns:
            bool: успех операции
        """
        try:
            # Парсим XML
            tree = ET.parse(filename)
            root = tree.getroot()
            
            # Очищаем старые данные
            self.data = []
            self.array_data = {}
            
            # Парсим устройство
            count = 0
            for device in root.findall('device'):
                for param in device:
                    at = param.get('name', '')
                    tp = param.get('type', '')
                    caption = param.get('caption', '')
                    
                    # Проверяем, является ли параметр массивом
                    match = re.search(r'^(.+)\[(\d+)\]$', at)
                    
                    if match:
                        # Это элемент массива
                        array_name = match.group(1)
                        index = int(match.group(2))
                        is_array = "Да"
                        
                        # Сохраняем в словарь массивов
                        if array_name not in self.array_data:
                            self.array_data[array_name] = {
                                'name': array_name,
                                'type': tp,
                                'elements': {}
                            }
                        self.array_data[array_name]['elements'][index] = {
                            'full_name': at,
                            'type': tp,
                            'caption': caption
                        }
                    else:
                        # Обычный параметр
                        is_array = "Нет"
                    
                    # Определяем размер типа
                    size = self.get_type_size(tp)
                    
                    # Создаем запись
                    item = {
                        'count': count,
                        'class': device.tag,
                        'name': at,
                        'type': tp,
                        'caption': caption,
                        'value': 0,
                        'size': size,
                        'is_array': is_array,
                        'array_name': match.group(1) if match else None,
                        'array_index': int(match.group(2)) if match else None
                    }
                    self.data.append(item)
                    count += 1
            
            # Заполняем таблицу
            self.fill_table()
            
            # Обновляем информацию
            self.count_label.config(text=f"Записей: {len(self.data)}")
            
            # Показываем информацию о массивах
            if self.array_data:
                arrays_info = f"Найдено массивов: {len(self.array_data)}"
                self.detail_label.config(text=arrays_info, fg="green")
            
            return True
            
        except ET.ParseError as e:
            messagebox.showerror("Ошибка парсинга", f"Неверный формат XML:\n{str(e)}")
            return False
        except Exception as e:
            messagebox.showerror("Ошибка", f"Не удалось загрузить файл:\n{str(e)}")
            return False
    
    def get_type_size(self, tp):
        """Определение размера типа данных"""
        type_sizes = {
            'unsigned char': 1,
            'signed char': 1,
            'unsigned int': 2,
            'signed int': 2,
            'unsigned long': 4,
            'signed long': 4,
            'float': 4,
            'double': 8,
        }
        return type_sizes.get(tp, 0)
    
    def fill_table(self):
        """Заполнение таблицы данными"""
        # Очищаем таблицу
        for item in self.tree.get_children():
            self.tree.delete(item)
        
        # Заполняем данными
        for row in self.data:
            values = [
                row.get('count', ''),
                row.get('class', ''),
                row.get('name', ''),
                row.get('type', ''),
                row.get('caption', ''),
                row.get('value', ''),
                row.get('is_array', '')
            ]
            self.tree.insert('', tk.END, values=values)
    
    def refresh_table(self):
        """Обновление таблицы"""
        if self.current_file:
            self.parse_xml(self.current_file)
        else:
            messagebox.showinfo("Информация", "Сначала выберите XML файл")
    
    def clear_all(self):
        """Очистка всех данных"""
        # Очищаем таблицу
        for item in self.tree.get_children():
            self.tree.delete(item)
        
        # Очищаем данные
        self.data = []
        self.array_data = {}
        self.current_file = None
        
        # Обновляем статус
        self.status_label.config(text="Очищено", fg="blue")
        self.info_label.config(text="Файл не загружен")
        self.count_label.config(text="Записей: 0")
        self.detail_label.config(text="")
    
    def export_csv(self):
        """Экспорт данных в CSV"""
        if not self.data:
            messagebox.showwarning("Предупреждение", "Нет данных для экспорта")
            return
        
        filename = filedialog.asksaveasfilename(
            title="Сохранить как CSV",
            defaultextension=".csv",
            filetypes=[
                ("CSV files", "*.csv"),
                ("All files", "*.*")
            ]
        )
        
        if filename:
            try:
                import csv
                with open(filename, 'w', newline='', encoding='utf-8-sig') as f:
                    writer = csv.writer(f)
                    
                    # Заголовки
                    headers = ['№', 'Класс', 'Имя', 'Тип', 'Описание', 'Значение', 'Массив']
                    writer.writerow(headers)
                    
                    # Данные
                    for row in self.data:
                        writer.writerow([
                            row.get('count', ''),
                            row.get('class', ''),
                            row.get('name', ''),
                            row.get('type', ''),
                            row.get('caption', ''),
                            row.get('value', ''),
                            row.get('is_array', '')
                        ])
                
                messagebox.showinfo("Успех", f"Данные сохранены в:\n{filename}")
                
            except Exception as e:
                messagebox.showerror("Ошибка", f"Не удалось сохранить:\n{str(e)}")
    
    def on_item_select(self, event):
        """Обработка выбора строки в таблице"""
        selected = self.tree.selection()
        if selected:
            values = self.tree.item(selected[0], 'values')
            if values:
                name = values[2]
                tp = values[3]
                caption = values[4]
                
                # Обновляем детальную информацию
                detail = f"Выбран: {name} (тип: {tp}"
                if caption:
                    detail += f", описание: {caption}"
                detail += ")"
                self.detail_label.config(text=detail, fg="blue")

# ============================================
# ЗАПУСК ПРИЛОЖЕНИЯ
# ============================================

if __name__ == "__main__":
    root = tk.Tk()
    app = XMLTableApp(root)
    root.mainloop()

In [8]:
#================================================================================================================================================================================
#                                                                                                                          Отправка и принятие ответа устройства
#================================================================================================================================================================================

import serial
from serial.rs485 import RS485
import struct

ser = RS485(port='COM5', baudrate=57600, timeout=1) #'/dev/ttyUSB0'
# Включаем режим RS485, RTS поднимается перед отправкой и опускается после
ser.rs485_mode = serial.rs485.RS485Settings(rts_level_for_tx=True, 
                                            rts_level_for_rx=False)
size = 0
command = bytearray([0x55, 0x55, 0x12, 0x01, 0x84, 0x05, 0x00, 0x08, 0x00, 0x00, 0xFE, 0x00])

def calculate_crc(data):
    
    """Расчет CRC16 для Modbus RTU"""
    crc = 0x0000
    for byte in data:
        crc += byte
    crc -= 0xAA  
    print(crc)
    return crc
    
crc = calculate_crc(command)    
command.append(crc & 0xFF)    

size = command[10]
ser.write(command)                                               #Отправка команды
print(str(command[11])+ str(command[10]))
receive_data = ser.read(size+8) 
data_reg = receive_data[7:7+size]
#print(len(part1))
print("Сырые байты:", data_reg.hex(' '))          # Выведет b'\x02\x05\xb6...'
print("В виде чисел:", list(receive_data))   # Выведет [2, 5, 182, ...]
print("Шестнадцатерично:", receive_data.hex(' ')) # Выведет '0205b6...'
# Работаем как обычно
float_value = round(struct.unpack('<f', data_reg[27:31])[0],2)
print(data_reg[27:31].hex(' '))#(float_value)
print(float_value)
ser.close()
#================================================================================================================================================================================
#                                                                                                                 Парсинг XML файла и отрисовка таблицы
#================================================================================================================================================================================
import tkinter as tk
from tkinter import ttk, filedialog, messagebox
import xml.etree.ElementTree as ET
import re

class XMLTableApp:
    """Приложение для загрузки XML и отображения в таблице"""
    
    def __init__(self, root):
        self.root = root
        self.root.title("XML Viewer с выбором файла")
        self.root.geometry("1000x700")
        
        # Переменные
        self.current_file = None
        self.data = []
        self.tree = None
        
        # Создаем интерфейс
        self.create_widgets()
        
        # Статус
        self.status_label.config(text="Загрузите XML файл", fg="blue")
    
    def create_widgets(self):
        """Создание всех виджетов"""
        # ============================================
        # Верхняя панель с кнопками
        # ============================================
        toolbar = tk.Frame(self.root)
        toolbar.pack(fill=tk.X, padx=5, pady=5)
        
        # Кнопка выбора XML файла
        self.load_btn = tk.Button(
            toolbar, 
            text="📁 Выбрать XML файл", 
            command=self.load_xml_file,
            bg="#4CAF50", 
            fg="white",
            font=('Arial', 10, 'bold'),
            padx=10,
            pady=5
        )
        self.load_btn.pack(side=tk.LEFT, padx=5)
        
        # Кнопка обновления
        self.refresh_btn = tk.Button(
            toolbar, 
            text="🔄 Обновить", 
            command=self.refresh_table,
            bg="#2196F3", 
            fg="white",
            padx=10,
            pady=5
        )
        self.refresh_btn.pack(side=tk.LEFT, padx=5)
        
        # Кнопка экспорта
        self.export_btn = tk.Button(
            toolbar, 
            text="📊 Экспорт CSV", 
            command=self.export_csv,
            bg="#FF9800", 
            fg="white",
            padx=10,
            pady=5
        )
        self.export_btn.pack(side=tk.LEFT, padx=5)
        
        # Кнопка очистки
        self.clear_btn = tk.Button(
            toolbar, 
            text="🗑️ Очистить", 
            command=self.clear_all,
            bg="#f44336", 
            fg="white",
            padx=10,
            pady=5
        )
        self.clear_btn.pack(side=tk.LEFT, padx=5)
        
        # Статус
        self.status_label = tk.Label(
            toolbar, 
            text="Готов", 
            font=('Arial', 10),
            fg="green"
        )
        self.status_label.pack(side=tk.RIGHT, padx=10)
        
        # ============================================
        # Информационная панель
        # ============================================
        info_frame = tk.Frame(self.root)
        info_frame.pack(fill=tk.X, padx=10, pady=5)
        
        self.info_label = tk.Label(
            info_frame, 
            text="Файл не загружен", 
            font=('Arial', 9),
            fg="gray"
        )
        self.info_label.pack(side=tk.LEFT)
        
        self.count_label = tk.Label(
            info_frame, 
            text="Записей: 0", 
            font=('Arial', 9),
            fg="gray"
        )
        self.count_label.pack(side=tk.RIGHT)
        
        # ============================================
        # Таблица (Treeview)
        # ============================================
        table_frame = tk.Frame(self.root)
        table_frame.pack(fill=tk.BOTH, expand=True, padx=10, pady=5)
        
        self.create_table(table_frame)
        
        # ============================================
        # Нижняя панель с дополнительной информацией
        # ============================================
        bottom_frame = tk.Frame(self.root)
        bottom_frame.pack(fill=tk.X, padx=10, pady=5)
        
        self.detail_label = tk.Label(
            bottom_frame, 
            text="", 
            font=('Arial', 9),
            fg="gray"
        )
        self.detail_label.pack(side=tk.LEFT)
    
    def create_table(self, parent):
        """Создание таблицы Treeview"""
        # Определяем колонки
        self.columns = ('count', 'class', 'name', 'type', 'caption', 'value')
        column_labels = {
            'count': '№',
            'class': 'Класс',
            'name': 'Имя',
            'type': 'Тип',
            'caption': 'Описание',
            'value': 'Значение'
        }
        
        # Создаем Treeview
        self.tree = ttk.Treeview(parent, columns=self.columns, show='headings')
        
        # Настраиваем колонки
        for col in self.columns:
            self.tree.heading(col, text=column_labels.get(col, col))
            width = 50 if col == 'count' else 150
            self.tree.column(col, width=width, anchor='center')
        
        # Увеличиваем ширину для некоторых колонок
        self.tree.column('caption', width=200)
        self.tree.column('name', width=150)
        
        # Добавляем скролл
        scrollbar = ttk.Scrollbar(parent, orient=tk.VERTICAL, command=self.tree.yview)
        self.tree.configure(yscrollcommand=scrollbar.set)
        
        # Размещаем
        self.tree.pack(side=tk.LEFT, fill=tk.BOTH, expand=True)
        scrollbar.pack(side=tk.RIGHT, fill=tk.Y)
        
        # Привязываем событие выбора строки
        self.tree.bind('<<TreeviewSelect>>', self.on_item_select)
    
    def load_xml_file(self):
        """Загрузка XML файла через диалоговое окно"""
        filename = filedialog.askopenfilename(
            title="Выберите XML файл",
            filetypes=[
                ("XML files", "*.xml"),
                ("All files", "*.*")
            ],
            initialdir="./"
        )
        
        if filename:
            self.current_file = filename
            self.status_label.config(text=f"Загрузка: {filename}", fg="orange")
            self.info_label.config(text=f"Файл: {filename}")
            self.root.update()
            
            # Парсим и заполняем таблицу
            success = self.parse_xml(filename)
            
            if success:
                self.status_label.config(text=f"✅ Загружен: {filename}", fg="green")
            else:
                self.status_label.config(text="❌ Ошибка загрузки", fg="red")
    
    def parse_xml(self, filename):
        """
        Парсинг XML файла с вашим алгоритмом
        Теперь ищем 'item' вместо 'device'
        """
        try:
            # Парсим XML
            tree = ET.parse(filename)
            root = tree.getroot()
            
            print(f"\nКорневой тег: {root.tag}")
            
            # ============================================
            # ИЩЕМ ЭЛЕМЕНТЫ 'item' (ВМЕСТО 'device')
            # ============================================
            elements = root.findall('item')
            
            # Если item не найден, проверяем все дочерние элементы
            if not elements:
                print("⚠️ Элементы 'item' не найдены")
                # Пробуем найти все элементы
                all_children = list(root)
                if all_children:
                    print(f"Найдены элементы: {[e.tag for e in all_children]}")
                    # Используем первый элемент как контейнер
                    elements = all_children
                    print("Используем все дочерние элементы как контейнеры")
            
            print(f"Найдено контейнеров: {len(elements)}")
            
            # Очищаем данные
            self.data = []
            
            # ============================================
            # ВАШ АЛГОРИТМ
            # ============================================
            count = 0
            m = 0
            
            for j in range(len(elements)):
                #print(f"\nЭлемент {j}: {elements[j].tag}")
                #print(f"  Дочерних: {len(elements[j])}")
                
                # Проходим по всем дочерним элементам
                for k in range(len(elements[j])):
                    child = elements[j][k]
                    at = child.get('name')
                    tp = child.get('type')
                    caption = child.get('caption', '')
                    
                    #print(f"  Параметр {k}: name='{at}', type='{tp}'")
                    
                    if at is None:
                        print(f"    ⚠️ Пропуск: name отсутствует")
                        continue
                    
                    # Поиск массива
                    match = re.search(r'^(.+)\[(\d+)\]$', at)
                    
                    if match:
                        # Это элемент массива
                        array_name = match.group(1)
                        index = int(match.group(2))
                        #print(f"    ✅ Массив: {array_name}[{index}]")
                        
                        for i in range(index):
                            item = {
                                "count": count,
                                "class": elements[j].tag,
                                "name": f"{array_name}[{i}]",
                                "type": tp,
                                "caption": caption,
                                "value": 0
                            }
                            self.data.append(item)
                            m += 1
                            
                            # Увеличиваем счетчик адресов
                            if tp in ('unsigned char', 'signed char'):
                                count += 1
                            elif tp in ('unsigned long', 'float'):
                                count += 4
                            elif tp == 'unsigned int':
                                count += 2
                            
                            #print(f"      Добавлен: {item['name']}, адрес: {item['count']}")
                    else:
                        # Обычный параметр
                        item = {
                            "count": count,
                            "class": elements[j].tag,
                            "name": at,
                            "type": tp,
                            "caption": caption,
                            "value": 0
                        }
                        self.data.append(item)
                        m += 1
                        
                        # Увеличиваем счетчик адресов
                        if tp in ('unsigned char', 'signed char'):
                            count += 1
                        elif tp in ('unsigned long', 'float'):
                            count += 4
                        elif tp == 'unsigned int':
                            count += 2
                        
                        #print(f"    Добавлен: {item['name']}, адрес: {item['count']}")
            offset = 0  # Текущее смещение в байтах

            for d in self.data:                                                               #data_reg = receive_data[7:7+size]
                if d['type'] in ('unsigned char', 'signed char'):
                    
                    d['value'] = data_reg[offset:offset+1]
                    offset += 1  # 1 байт
                elif d['type'] == 'unsigned long':
                    if len(data_reg) >= offset + 4:
                        d['value'] = struct.unpack('<I', data_reg[offset:offset+4])[0]
                    else:
                        print(f"Предупреждение: недостаточно данных для float по адресу {offset}")
                        d['value'] = "NONI"  # Или None, или другое значение по умолчанию

                
                    #d['value'] = struct.unpack('<I', data_reg[offset:offset+4])[0] #data_reg[offset:offset+4].hex()
                    #offset += 4  # 4 байта
                elif d['type'] == 'float':
                    
                    if len(data_reg) >= offset + 4:
                        d['value'] = struct.unpack('<f', data_reg[offset:offset+4])[0]
                    else:
                        print(f"Предупреждение: недостаточно данных для float по адресу {offset}")
                        d['value'] = "NONI"  # Или None, или другое значение по умолчанию
                    #float_data = struct.unpack('<f', data_reg[offset:offset+4])[0]
                    #d['value'] = float_data if offset < 255 else "NONI" #struct.unpack('<f', bytes.fromhex(data_reg[offset:offset+4].hex()))  round(struct.unpack('<f', data_reg[offset:offset+4])[0],2)
                    offset += 4  # 4 байта    
                elif d['type'] in ('unsigned int', ):
                    d['value'] = data_reg[offset:offset+2].hex()
                    offset += 2  # 2 байта 
                                                                                          #========================================================
                

            
            
            print(f"\n" + "="*60)
            print(f"ПАРСИНГ ЗАВЕРШЕН")
            print(f"Всего записей: {len(self.data)}")
            print(f"Последний адрес: {count}")
            print("="*60)
            
            # ============================================
            # ЗАПОЛНЯЕМ ТАБЛИЦУ
            # ============================================
            self.fill_table()
            
            # Обновляем информацию
            self.count_label.config(text=f"Записей: {len(self.data)}")
            
            return True
            
        except ET.ParseError as e:
            print(f"❌ Ошибка парсинга: {e}")
            messagebox.showerror("Ошибка парсинга", f"Неверный формат XML:\n{str(e)}")
            return False
        except Exception as e:
            print(f"❌ Ошибка: {e}")
            import traceback
            traceback.print_exc()
            messagebox.showerror("Ошибка", f"Не удалось загрузить файл:\n{str(e)}")
            return False
    
    def fill_table(self):
        """Заполнение таблицы данными"""
        print(f"\nЗАПОЛНЕНИЕ ТАБЛИЦЫ: {len(self.data)} записей")
        
        # Очищаем таблицу
        for item in self.tree.get_children():
            self.tree.delete(item)
        
        # Проверяем данные
        if not self.data:
            print("⚠️ Нет данных для заполнения")
            self.status_label.config(text="Нет данных", fg="orange")
            return
        
        # Заполняем данными
        for row in self.data:
            values = [
                row.get('count', ''),
                row.get('class', ''),
                row.get('name', ''),
                row.get('type', ''),
                row.get('caption', ''),
                row.get('value', '')
            ]
            self.tree.insert('', tk.END, values=values)
        
        print(f"✅ Добавлено {len(self.data)} строк")
        self.status_label.config(text=f"Загружено {len(self.data)} записей", fg="green")
    
    def refresh_table(self):
        """Обновление таблицы"""
        if self.current_file:
            self.parse_xml(self.current_file)
        else:
            messagebox.showinfo("Информация", "Сначала выберите XML файл")
    
    def clear_all(self):
        """Очистка всех данных"""
        # Очищаем таблицу
        for item in self.tree.get_children():
            self.tree.delete(item)
        
        # Очищаем данные
        self.data = []
        self.current_file = None
        
        # Обновляем статус
        self.status_label.config(text="Очищено", fg="blue")
        self.info_label.config(text="Файл не загружен")
        self.count_label.config(text="Записей: 0")
        self.detail_label.config(text="")
    
    def export_csv(self):
        """Экспорт данных в CSV"""
        if not self.data:
            messagebox.showwarning("Предупреждение", "Нет данных для экспорта")
            return
        
        filename = filedialog.asksaveasfilename(
            title="Сохранить как CSV",
            defaultextension=".csv",
            filetypes=[
                ("CSV files", "*.csv"),
                ("All files", "*.*")
            ]
        )
        
        if filename:
            try:
                import csv
                with open(filename, 'w', newline='', encoding='utf-8-sig') as f:
                    writer = csv.writer(f)
                    
                    # Заголовки
                    headers = ['№', 'Класс', 'Имя', 'Тип', 'Описание', 'Значение']
                    writer.writerow(headers)
                    
                    # Данные
                    for row in self.data:
                        writer.writerow([
                            row.get('count', ''),
                            row.get('class', ''),
                            row.get('name', ''),
                            row.get('type', ''),
                            row.get('caption', ''),
                            row.get('value', '')
                        ])
                
                messagebox.showinfo("Успех", f"Данные сохранены в:\n{filename}")
                
            except Exception as e:
                messagebox.showerror("Ошибка", f"Не удалось сохранить:\n{str(e)}")
    
    def on_item_select(self, event):
        """Обработка выбора строки в таблице"""
        selected = self.tree.selection()
        if selected:
            values = self.tree.item(selected[0], 'values')
            if values:
                name = values[2]
                tp = values[3]
                caption = values[4]
                addr = values[0]
                
                # Обновляем детальную информацию
                detail = f"Адрес: {addr}, Имя: {name}, Тип: {tp}"
                if caption:
                    detail += f", Описание: {caption}"
                self.detail_label.config(text=detail, fg="blue")

# ============================================
# ЗАПУСК ПРИЛОЖЕНИЯ
# ============================================

if __name__ == "__main__":
    root = tk.Tk()
    app = XMLTableApp(root)
    root.mainloop()

418
0254
Сырые байты: 06 00 0f 00 02 00 00 d5 1f 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 51 45 80 41 00 00 00 00 cb 3a 32 40 22 a0 88 40 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 35 ea 9b 3f 94 ff 0f 41 98 02 82 41 b8 4a 07 41 e8 4c 2d 41 98 02 82 41 b8 4a 07 41 e8 4c 2d 41 00 00 00 00 00 00 00 00 35 ea 9b 3f 94 ff 0f 41 c8 ec 29 40 58 27 28 40 00 90 26 40 00 00 04 ac 50 00 00 ac 50 00 00 00 00 00 00 01 00 00 00 ff ff 00 00 00 00 00 00 00 00 55 55 55 55 d5 40 00 00 00 00 34 0c 00 00 ff ff ff ff ff ff 38 32 2b 01 1a 01 00 00 10 5f 28 01 23 03 00 00 60 cc 02 00 60 03 00 00 26 05 00 00 25 05 00 00 08 00 00 00 07 06 00 00 00 00 00 00 42 60 e5 3e 00 00 00 00 00 00 00 00 37 03 00 00 2e 02 00 00 1f 02 00 00 1a 02 00 00 15 02 00 00 00 00 00 00 00 00 00 00 38 a1
В виде чисел: [85, 85, 1, 18, 132, 254, 0, 6, 0, 15, 0, 2, 0, 0, 213, 31, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 81, 69, 128, 65, 0, 0, 0, 0, 203, 58, 50, 64, 34, 160, 136, 64, 0, 0, 0, 0, 0, 0, 0, 0, 